# KV260 — Phương án A: độ trễ và năng lượng SUY LUẬN của `ecg_f2a_top` (bài journal BSPC)
**Notebook:** `kv260_f2a_inference_perf.ipynb` — tạo 19/10/2026. Đo các số còn thiếu cho bài journal, **cùng giao thức với bài ICMLANT** (bản ROM `ecg_f2_top`) để so sánh trực tiếp.

| Mục | Đo gì | Giao thức |
|---|---|---|
| 1 | Bit-exact T1/T3 với C-sim (1 000 nhịp) | như `kv260_ecg_f2a.ipynb` |
| 2 | Độ trễ F2A: *invocation* (ap_start → ap_done) và *AXI-inclusive* (ghi đầu vào → đọc logit + nhãn); thêm chế độ **có đọc emb** (pha hiệu chỉnh) và thời gian **ghi lớp đầu ra** | 50 lần khởi động, **trung vị 300 lần** |
| 3 | Thông lượng toàn DS2 (49 649 nhịp) + kiểm tra acc = 0,9401 | 1 lượt |
| 4 | Công suất INA260 (1 kênh), năng lượng tăng thêm/nhịp: F2A suy luận, F2A suy luận + đọc emb, ARM float32 theo lô | khối 3 000 nhịp (ARM: 2 000) lặp ≥ 15 s, mẫu 50 ms, nền 10 s trước **và** sau |
| 5 | (tùy chọn) **Bản ROM `ecg_f2_top` đo lại trong cùng buổi** + so logit F2 với F2A | như notebook ICMLANT |

**File cần có trong thư mục chạy (`F2A_BOARD_RUN`, phần lớn đã có từ các lần trước):**
`ecg_f2a.bit`, `ecg_f2a.hwh`, `xecg_f2a_top_hw.h`, `lib_ecg_f2a.py`, `lib_ecg_f2.py`, `pa_adapt.py`, `weights_f2a.h`, `f2_params.npz`,
`test_beats_q.txt`, `test_rr_q.txt`, `test_logits_sim.txt`, `test_emb_sim.txt`, `test_head_adapted.txt`, `test_logits_adapted.txt`, `f2a_board_data.npz`.
Cho mục 5 (tùy chọn): tạo thư mục con `f2_rom/` và chép vào đó `ecg_f2.bit`, `ecg_f2.hwh`, `xecg_f2_top_hw.h` từ `04_vivado/03_ecg_f2_seed3/` của kho.

**Trước khi chạy:** mở terminal trên board, chạy
`sudo systemctl stop unattended-upgrades apt-daily.timer apt-daily-upgrade.timer`, đóng mọi notebook/terminal khác, rồi **Run All**. Trong lúc chạy mục 4–5 không thao tác gì trên board. Tổng thời gian khoảng 6–8 phút.

**Kết quả:** `f2a_inference_perf.json` (gửi lại file này) — chép về kho tại `05_board/ecg_f2a_phaseA/`.

In [ ]:
import os, io, sys, json, time, platform, contextlib, warnings
import numpy as np
warnings.filterwarnings('ignore')                       # bỏ cảnh báo libsensors của PYNQ (vô hại)
from pynq import Overlay, Clocks
import lib_ecg_f2a as L, lib_ecg_f2 as LF, pa_adapt as pa

NEED = ['ecg_f2a.bit', 'ecg_f2a.hwh', 'xecg_f2a_top_hw.h', 'lib_ecg_f2a.py', 'lib_ecg_f2.py', 'pa_adapt.py', 'weights_f2a.h',
        'f2_params.npz', 'test_beats_q.txt', 'test_rr_q.txt', 'test_logits_sim.txt', 'test_emb_sim.txt',
        'test_head_adapted.txt', 'test_logits_adapted.txt', 'f2a_board_data.npz']
miss = [f for f in NEED if not os.path.exists(f)]
print('Thư mục:', os.getcwd()); print('Thiếu:', miss or 'không')
assert not miss, 'Chép đủ file trước khi chạy'
RUN_F2 = all(os.path.exists(os.path.join('f2_rom', f)) for f in ['ecg_f2.bit', 'ecg_f2.hwh'])
print('Mục 5 (đo lại bản ROM trong cùng buổi):', 'CÓ' if RUN_F2 else 'BỎ QUA (không thấy f2_rom/ecg_f2.bit/.hwh)')

RES = dict(notebook='kv260_f2a_inference_perf.ipynb', run_at=time.strftime('%Y-%m-%d %H:%M:%S %z'),
           host=dict(kernel=platform.release(), python=platform.python_version(), numpy=np.__version__))
try:
    import pynq; RES['host']['pynq'] = pynq.__version__
except Exception:
    pass

ol = Overlay('ecg_f2a.bit')
print('fclk0 trước:', Clocks.fclk0_mhz, 'MHz')
Clocks.fclk0_mhz = 100                                  # thiết kế đóng timing ở 100 MHz; PYNQ mặc định 150 MHz
FCLK = float(Clocks.fclk0_mhz); print('fclk0 sau  :', FCLK, 'MHz'); RES['fclk_mhz'] = FCLK
assert abs(FCLK - 100) < 0.5, 'Clock PL phải là 100 MHz'
regs = L.parse_hw_header('xecg_f2a_top_hw.h')
ip = L.ECGF2A(ol.ecg_f2a_top_0.mmio, regs)
W_G, B_G = L.parse_global_head('weights_f2a.h'); ip.write_head(W_G, B_G)
w_rb, b_rb = ip.read_head(); ok = bool((w_rb == W_G).all() and (b_rb == B_G).all())
print('Đọc lại head_w/head_b khớp:', ok); RES['head_readback_ok'] = ok

## 1. Bit-exact với C-sim (T1: lớp đầu ra toàn cục, T3: lớp đầu ra đã hiệu chỉnh)

In [ ]:
B = np.loadtxt('test_beats_q.txt', dtype=np.int64); R = np.loadtxt('test_rr_q.txt', dtype=np.int64)
ip.write_head(W_G, B_G)
LG1, EM1, LAB1 = ip.infer_batch(B, R)
t1_lg = bool((LG1 == np.loadtxt('test_logits_sim.txt', dtype=np.int64)).all())
t1_em = bool((EM1 == np.loadtxt('test_emb_sim.txt', dtype=np.int64)).all())
H = np.loadtxt('test_head_adapted.txt', dtype=np.int64); ip.write_head(H[:128], H[128:])
LG3, _, _ = ip.infer_batch(B, R)
t3 = bool((LG3 == np.loadtxt('test_logits_adapted.txt', dtype=np.int64)).all())
ip.write_head(W_G, B_G)
print(f'[T1] logit khớp: {t1_lg} | emb khớp: {t1_em} | [T3] logit khớp: {t3}  ({len(B)} nhịp)')
RES['bitexact'] = dict(n=len(B), T1_logits=t1_lg, T1_emb=t1_em, T3_logits=t3)
assert t1_lg and t1_em and t3, 'Board KHÔNG khớp C-sim -> kiểm tra bitstream/file test'

## 2. Độ trễ (giao thức ICMLANT: 50 lần khởi động, trung vị 300 lần)
- *invocation*: từ ghi `ap_start` tới khi thấy `ap_done` (bit 1) bằng polling — giống `lib_ecg_f2.EcgF2.classify`.
- *AXI-inclusive*: từ lần ghi đầu vào đầu tiên tới khi đọc xong logit và nhãn (chế độ **suy luận**), hoặc thêm 32 byte emb (chế độ **hiệu chỉnh**, dùng trong 5 phút đầu).
- Đầu vào được đóng gói sẵn thành từ 32 bit trước khi đo (như `classify_many` của bản ROM).

In [ ]:
MM = ip.mmio; ARR = MM.array; r_ = regs
o_ecg, o_rr, o_ctl = r_['ECG_IN_BASE'] // 4, r_['RR_IN_BASE'] // 4, r_['AP_CTRL']
o_lg, o_emb, o_lab = r_['LOGITS_BASE'] // 4, r_['EMB_OUT_BASE'] // 4, r_['LABEL_DATA']

def prep(beats, rrs):
    return [L._pack_i8(b) for b in beats], [L._pack_i8(r) for r in rrs]

def run_one(pb, pr, read_emb=False):
    # Trả (logits, emb|None, label, t_invocation_s, t_axi_s)
    s = time.perf_counter()
    ARR[o_ecg:o_ecg + len(pb)] = pb
    ARR[o_rr:o_rr + len(pr)] = pr
    t0 = time.perf_counter()
    MM.write(o_ctl, 1)
    while not (MM.read(o_ctl) & 0x2):
        pass
    t1 = time.perf_counter()
    lg = np.array(ARR[o_lg:o_lg + 4], dtype=np.uint32).view(np.int32)
    em = np.array(ARR[o_emb:o_emb + 8], dtype=np.uint32).view(np.int8) if read_emb else None
    lab = MM.read(o_lab) & 0x3
    return lg, em, lab, t1 - t0, time.perf_counter() - s

PB, PR = prep(B, R)
# đối chiếu nhanh: hàm đo cho cùng logit/emb với driver (100 nhịp)
for k in range(100):
    lg, em, lab, _, _ = run_one(PB[k], PR[k], True)
    assert (lg == LG1[k]).all() and (em == EM1[k]).all() and lab == LAB1[k]

def latency(read_emb, n_warm=50, n=300):
    for k in range(n_warm):
        run_one(PB[k], PR[k], read_emb)
    tc, ta = np.zeros(n), np.zeros(n)
    for k in range(n):
        _, _, _, tc[k], ta[k] = run_one(PB[k], PR[k], read_emb)
    return dict(invocation_ms=float(np.median(tc) * 1e3), axi_inclusive_ms=float(np.median(ta) * 1e3),
                invocation_p95_ms=float(np.percentile(tc, 95) * 1e3), axi_p95_ms=float(np.percentile(ta, 95) * 1e3),
                invocation_mean_ms=float(tc.mean() * 1e3), axi_mean_ms=float(ta.mean() * 1e3),
                host_cycles=int(round(np.median(tc) * FCLK * 1e6)), n=n, warmup=n_warm)

LAT = dict(infer=latency(False), infer_emb=latency(True))
# thời gian ghi lớp đầu ra (36 từ 32 bit: 128 int8 + 4 x Q18.14) — một lần mỗi bản ghi
for _ in range(50): ip.write_head(W_G, B_G)
tw = []
for _ in range(300):
    s = time.perf_counter(); ip.write_head(W_G, B_G); tw.append(time.perf_counter() - s)
LAT['write_head_us'] = dict(median=float(np.median(tw) * 1e6), p95=float(np.percentile(tw, 95) * 1e6))
RES['latency'] = LAT
for k in ['infer', 'infer_emb']:
    v = LAT[k]
    print(f"{k:10s} invocation {v['invocation_ms']:.3f} ms ({v['host_cycles']} chu kỳ) | AXI-inclusive {v['axi_inclusive_ms']:.3f} ms "
          f"(p95 {v['axi_p95_ms']:.3f})")
print(f"ghi lớp đầu ra: {LAT['write_head_us']['median']:.1f} µs (p95 {LAT['write_head_us']['p95']:.1f})")
print('Tham chiếu ICMLANT (ROM, seed 3): invocation 0.820 ms, AXI-inclusive 0.952 ms; co-sim F2A 82 211 chu kỳ')

## 3. Thông lượng trên toàn DS2 và kiểm tra độ chính xác (lớp đầu ra toàn cục)

In [ ]:
D = np.load('f2a_board_data.npz')
beats, rrs, y = D['beats_q'], D['rr_q'], D['y']
ip.write_head(W_G, B_G)
t0 = time.perf_counter()
pb_all, pr_all = prep(beats, rrs)
t_prep = time.perf_counter() - t0
pred = np.zeros(len(y), int)
t1 = time.perf_counter()
for k in range(len(y)):
    pred[k] = run_one(pb_all[k], pr_all[k])[2]
t_run = time.perf_counter() - t1
acc = float((pred == y).mean())
RES['ds2_throughput'] = dict(beats=len(y), seconds_run=t_run, seconds_prep=t_prep, ms_per_beat=t_run / len(y) * 1e3,
                             beats_per_s=len(y) / t_run, acc_global_head=acc)
print(f'{len(y)} nhịp: {t_run:.1f} s ({t_run / len(y) * 1e3:.3f} ms/nhịp, {len(y) / t_run:.0f} nhịp/s) + đóng gói {t_prep:.1f} s')
print(f'acc toàn DS2 (lớp đầu ra toàn cục): {acc:.5f}  (kỳ vọng 0.94010)')
assert abs(acc - 0.94010) < 5e-5, 'acc khác mô phỏng bit-exact'

## 4. Công suất và năng lượng tăng thêm mỗi nhịp (INA260 trên nguồn SOM, đếm **một** kênh)
Năng lượng tăng thêm = (P khi chạy − P rảnh) × thời gian / số nhịp; P rảnh = trung bình 10 s trước và 10 s sau.

In [ ]:
pm = LF.PowerMonitor(verbose=False)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    pm.list_sensors()
sensors = buf.getvalue().strip(); print('Kênh đo công suất:\n' + sensors)
RES['power_sensor'] = dict(mode=pm.mode, channels=[n for n, _, _ in pm.files], listing=sensors)
assert pm.mode == 'pynq' or len(pm.files) == 1, 'Có nhiều hơn 1 kênh -> kiểm tra trước khi đo (tránh cộng trùng)'

N_BLK = 3000
idx = np.arange(N_BLK)
def block_fpga(read_emb):
    def f():
        pb, pr = prep(beats[idx], rrs[idx])            # gồm cả đóng gói, như classify_many của bản ROM
        for k in range(N_BLK):
            run_one(pb[k], pr[k], read_emb)
    return f
nf = LF.NumpyF2('f2_params.npz')
Xf = beats[:2000].astype(np.float32) / 64.0; Rf = rrs[:2000].astype(np.float32) / 32.0   # chỉ để đo thời gian/năng lượng
nf.float32(Xf[:16], Rf[:16])

import gc
idle0 = pm.baseline(seconds=10)
ST = {}
for name, fn, n in [('fpga_infer', block_fpga(False), N_BLK), ('fpga_infer_emb', block_fpga(True), N_BLK),
                    ('arm_f32_batch', lambda: nf.float32(Xf, Rf), 2000)]:
    gc.collect(); time.sleep(2)
    _, st = pm.measure(fn, period=0.05, min_seconds=15)
    ST[name] = (st, n)
    print(f"{name:16s} {st['seconds']:.1f} s, {st['reps']} khối, P_mean {st['p_mean_w']:.3f} W, {st['n_samples']} mẫu")
idle1 = pm.baseline(seconds=10); idle = (idle0 + idle1) / 2

def energy(st, n):
    p_act = st['p_mean_w'] - idle; beats_done = n * st['reps']
    return dict(p_mean_w=st['p_mean_w'], p_active_w=p_act, seconds=st['seconds'], beats=beats_done,
                ms_per_beat=st['seconds'] / beats_done * 1e3, energy_active_mJ=p_act * st['seconds'] / beats_done * 1e3,
                n_samples=st['n_samples'])
PW = dict(idle_w=idle, idle_before_w=idle0, idle_after_w=idle1, **{k: energy(*v) for k, v in ST.items()})
PW['reduction_vs_arm_batch'] = 1 - PW['fpga_infer']['energy_active_mJ'] / PW['arm_f32_batch']['energy_active_mJ']
RES['power'] = PW
print(f"\nP rảnh {idle:.3f} W (trước {idle0:.3f}, sau {idle1:.3f})")
for k in ['fpga_infer', 'fpga_infer_emb', 'arm_f32_batch']:
    v = PW[k]; print(f"{k:16s} +{v['p_active_w']:.3f} W | {v['ms_per_beat']:.3f} ms/nhịp | {v['energy_active_mJ']:.3f} mJ/nhịp")
print(f"Giảm năng lượng so với ARM float32 theo lô: {PW['reduction_vs_arm_batch'] * 100:.1f} %")
print('Tham chiếu ICMLANT (ROM, seed 3): FPGA 0.230 mJ/nhịp, ARM lô 1.588 mJ/nhịp')

## 5. (Tùy chọn) Bản ROM `ecg_f2_top` đo lại trong cùng buổi
Nạp `f2_rom/ecg_f2.bit` (thay overlay F2A), kiểm tra logit trùng F2A với lớp đầu ra toàn cục, rồi đo độ trễ và năng lượng bằng **đúng** hàm của notebook ICMLANT (`lib_ecg_f2.EcgF2.classify_many`).

In [ ]:
if RUN_F2:
    f2 = LF.EcgF2('f2_rom/ecg_f2.bit', header='f2_rom/xecg_f2_top_hw.h', fclk_mhz=100)
    lg_f2 = np.array([f2.classify(B[k], R[k])[0] for k in range(len(B))])
    same = bool((lg_f2 == LG1).all())
    print(f'Logit F2 (ROM) trùng F2A (lớp đầu ra toàn cục) trên {len(B)} nhịp: {same}')
    for i in range(50):
        f2.classify(B[i], R[i])
    r = f2.classify_many(B[:300], R[:300])
    lat_f2 = dict(invocation_ms=float(np.median(r['t_core_us']) / 1e3), axi_inclusive_ms=float(np.median(r['t_total_us']) / 1e3),
                  host_cycles=int(round(np.median(r['t_core_us']) * f2.fclk_mhz)))
    print(f"F2 ROM: invocation {lat_f2['invocation_ms']:.3f} ms | AXI-inclusive {lat_f2['axi_inclusive_ms']:.3f} ms")
    i0 = pm.baseline(seconds=10)
    gc.collect(); time.sleep(2)
    _, st = pm.measure(lambda: f2.classify_many(beats[:3000], rrs[:3000]), period=0.05, min_seconds=15)
    i1 = pm.baseline(seconds=10); idle_f2 = (i0 + i1) / 2
    e_f2 = (st['p_mean_w'] - idle_f2) * st['seconds'] / (3000 * st['reps']) * 1e3
    print(f"F2 ROM: +{st['p_mean_w'] - idle_f2:.3f} W | {e_f2:.3f} mJ/nhịp (P rảnh {idle_f2:.3f} W)")
    RES['f2_rom_same_session'] = dict(logits_equal_f2a=same, latency=lat_f2, idle_w=idle_f2,
                                      p_active_w=st['p_mean_w'] - idle_f2, energy_active_mJ=e_f2,
                                      seconds=st['seconds'], beats=3000 * st['reps'])
else:
    print('Bỏ qua mục 5.')

## 6. Lưu kết quả và tóm tắt cho bài

In [ ]:
json.dump(RES, open('f2a_inference_perf.json', 'w'), indent=1, default=float, ensure_ascii=False)
L2, P2 = RES['latency'], RES['power']
print('=== TÓM TẮT (100 MHz, seed 3, F2A) ===')
print(f"Bit-exact T1/T3: {RES['bitexact']}")
print(f"Suy luận : invocation {L2['infer']['invocation_ms']:.3f} ms | AXI-inclusive {L2['infer']['axi_inclusive_ms']:.3f} ms | "
      f"{P2['fpga_infer']['energy_active_mJ']:.3f} mJ/nhịp (+{P2['fpga_infer']['p_active_w']:.3f} W)")
print(f"Hiệu chỉnh: AXI-inclusive có đọc emb {L2['infer_emb']['axi_inclusive_ms']:.3f} ms | "
      f"{P2['fpga_infer_emb']['energy_active_mJ']:.3f} mJ/nhịp | ghi lớp đầu ra {L2['write_head_us']['median']:.1f} µs")
print(f"ARM float32 lô: {P2['arm_f32_batch']['ms_per_beat']:.3f} ms/nhịp, {P2['arm_f32_batch']['energy_active_mJ']:.3f} mJ/nhịp "
      f"-> FPGA giảm {P2['reduction_vs_arm_batch'] * 100:.1f} %")
print(f"Toàn DS2: {RES['ds2_throughput']['ms_per_beat']:.3f} ms/nhịp, acc {RES['ds2_throughput']['acc_global_head']:.5f}")
if 'f2_rom_same_session' in RES:
    f = RES['f2_rom_same_session']
    print(f"ROM cùng buổi: {f['latency']['axi_inclusive_ms']:.3f} ms, {f['energy_active_mJ']:.3f} mJ/nhịp, logit trùng F2A: {f['logits_equal_f2a']}")
print('Đã lưu f2a_inference_perf.json -> gửi lại file này')